In [ ]:
import os
import random
import pickle
 
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.backends.cudnn as cudnn
from sklearn.cluster import KMeans

In [ ]:
def fix_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    cudnn.deterministic = True
    cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    torch.set_num_threads(1)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

fix_seed(42)

Load Datasets

In [ ]:
# SC datasets
# sampled_adata = sc.read_h5ad("data/human_spleen_updated.h5ad")
# sampled_adata = sc.read_h5ad("data/gse155468.h5ad")
sampled_adata = sc.read_h5ad("data/GSE194122_cite_BMMC_processed.h5ad")

# SRT datasets
# sampled_adata = sc.read_h5ad("data/sshippo.h5ad")
# sampled_adata = sc.read_h5ad("data/E9.5_E1S1.MOSTA.h5ad")
# sampled_adata = sc.read_h5ad("data/breast_cancer.h5ad")

dataset = "BMMC"
datatype = "sc"

In [ ]:
with open("data/ensem_emb_gpt3.5all_new.pickle", "rb") as fp:
    GPT_3_5_gene_embeddings = pickle.load(fp)

gene_names = np.array(sampled_adata.var.index)

embed_df = pd.DataFrame.from_dict(GPT_3_5_gene_embeddings, orient="index")
embed_df = embed_df.reindex(gene_names)
embed_df = embed_df.fillna(0.0)

lookup_embed = embed_df.values.astype(np.float32)

count_missing = np.sum(np.all(lookup_embed == 0, axis=1))
print(f"Unable to match {count_missing} out of {len(gene_names)} genes in the GenePT-w embedding")

genePT_w_embed = sampled_adata.X @ lookup_embed / len(gene_names)

In [ ]:
embeddings = genePT_w_embed

if dataset == "HS" or dataset == "BMMC":
    label_variable = "cell_type"
elif dataset == "MH":
    label_variable = "cluster"
elif dataset == "ME":
    label_variable = "annotation"
elif dataset == "BC":
    label_variable = "fine_annot_type"
else:
    label_variable = "celltype"


In [ ]:
def kmeans_clustering(dataset, embeddings, sampled_adata, label_variable):
    adata_tmp = sc.AnnData(embeddings)
    adata_tmp.obsm["GenePT-w"] = embeddings
    
    n_clusters = len(np.unique(sampled_adata.obs[label_variable]))
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)

    cluster_labels = kmeans.fit_predict(embeddings)
    adata_tmp.obs["KMeans_clusters"] = cluster_labels.astype(str)

    adata_tmp.obsm["spatial"] = sampled_adata.obsm["spatial"]

    print("Number of KMeans clusters:", adata_tmp.obs["KMeans_clusters"].nunique())

def leiden_clustering(dataset, embeddings, sampled_adata, resolution):
    adata_tmp = sc.AnnData(embeddings)
    adata_tmp.obsm["GenePT-w"] = embeddings

    sc.pp.neighbors(adata_tmp, use_rep="GenePT-w")
    sc.tl.leiden(adata_tmp, resolution=resolution, key_added="leiden_clusters")
    
    print("Number of Leiden clusters:", adata_tmp.obs["leiden_clusters"].nunique())

    adata_tmp.obsm["spatial"] = sampled_adata.obsm["spatial"]
    

In [ ]:
kmeans_clustering(dataset, embeddings, sampled_adata, label_variable)
leiden_clustering(dataset, embeddings, sampled_adata, resolution=1.0)